# Meeting → Document Pipeline (Audio → Transcript → Speaker Names → Executive Summary)

This notebook transcribes audio files, attributes speech to specific speakers, maps generic
speaker IDs to real names, and generates an executive summary document (DOCX / PDF) using a
local open-source LLM via Map-Reduce summarization.

## Pipeline stages
| Stage | What it does |
|---|---|
| 1. Diarize | `pyannote/speaker-diarization-3.1` finds speaker intervals |
| 2. Transcribe & Align | `openai-whisper` (`large-v3`) transcribes; segments are matched to speakers by maximum time overlap and merged into readable speaker turns |
| 3. Name mapping | Interactive rename of `SPEAKER_00` → real names |
| 4. Summarize | Map-Reduce summarization with `Qwen/Qwen2.5-7B-Instruct` running locally on the GPU |
| 5. Export | Markdown → Word (`.docx`) or PDF, plus raw transcript/summary saved to `reports/` |

## ROCm 10.1.0 setup — Windows 11 · Radeon RX 7900 XT (`gfx1100`) · pip
Source: [ROCm install selector](https://rocm.docs.amd.com/en/latest/install/rocm.html?fam=radeon&w=compute&gpu=amd-radeon-rx-7900-xt&os=windows&windows-ver=11&i=pip&gfx=gfx1100)

1. **Prepare Windows** (one-time per the AMD prerequisites): uninstall any HIP SDK / conflicting
   AMD graphics software, disable *Smart App Control* and *Windows Defender Application Guard*,
   and install the AMD Software: Adrenalin Edition driver.
2. **Create the virtual environment** (Python 3.12):
   ```bat
   py -3.12 -m venv .venv
   .venv\Scripts\activate
   python -m pip install ipykernel langgraph langchain-core langchain-community langchain-chroma \
       langchain-text-splitters langchain-huggingface sentence-transformers chromadb \
       "transformers<5.9.0" accelerate python-dotenv pypdf openai-whisper pyannote.audio \
       python-docx reportlab ipykernel
   python -m ipykernel install --user --name langGraph --display-name "langGraph"
   ```
3. **Install the ROCm core SDK** (pip wheels):
   ```bat
   python -m pip install --index-url https://stable.repo.amd.com/rocm/whl-next/ "rocm[libraries,device-gfx1100]==10.1.0"
   ```
4. **Install the ROCm-enabled PyTorch stack** (verified v2.14.0 wheels, gfx1100 kernels):
   ```bat
   python -m pip install --index-url https://stable.repo.amd.com/rocm/whl-next/ ^
       "torch[device-gfx1100]==2.14.0+rocm10.1.0" ^
       "torchvision[device-gfx1100]==0.29.0a0+rocm10.1.0" ^
       "torchaudio==2.11.0.3+rocm10.1.0"
   ```
   You can also verify the install with `hipinfo` and `rocm-sdk targets` in the same venv.
5. **Fix the broken `whisper` import** (the `TypeError: argument of type 'NoneType' is not
   iterable` crash): the single-file PyPI package `whisper` (a legacy Nautilus/NFS shim) crashes
   on Windows inside `ctypes.CDLL`. Remove it — **this is the crash that broke this notebook**:
   ```bat
   pip uninstall -y whisper
   pip install openai-whisper
   ```
6. **Models are fetched from HuggingFace and cached locally** under `models/`. The pyannote
   models are *gated*: log in at https://hf.co (accept the license for
   `pyannote/segmentation-3.1` and `pyannote/speaker-diarization-3.1`), then add
   `HF_TOKEN="hf_..."` to `config/.env`.
   After the first successful run you can switch `RUN_OFFLINE = True` in the config cell for a
   fully offline execution.

In [1]:
# ============================================================================
# 0) Pre-flight — fail fast with a clear explanation if the environment is bad
# ============================================================================
import importlib.util
import os

_OPENAI_WHISPER_INIT = os.path.join("whisper", "__init__.py").lower()

spec = importlib.util.find_spec("whisper")
if spec is None:
    raise ModuleNotFoundError(
        "openai-whisper is not installed.\n"
        "  Fix:  pip uninstall -y whisper && pip install openai-whisper"
    )
if not spec.origin.lower().endswith(_OPENAI_WHISPER_INIT):
    raise RuntimeError(
        f"\nFound the broken PyPI 'whisper' shim package instead of openai-whisper.\n"
        f"    Location: {spec.origin}\n"
        "This package crashes on Windows (ctypes.CDLL -> TypeError: argument of type\n"
        "'NoneType' is not iterable). Replace it with the real OpenAI Whisper:\n\n"
        "    pip uninstall -y whisper\n"
        "    pip install openai-whisper\n\n"
        "Then restart the Jupyter kernel and re-run."
    )
print(f"[Preflight] whisper package OK -> {spec.origin}")

RuntimeError: 
Found the broken PyPI 'whisper' shim package instead of openai-whisper.
    Location: c:\Users\lovep\miniconda3\envs\langGraph\Lib\site-packages\whisper.py
This package crashes on Windows (ctypes.CDLL -> TypeError: argument of type
'NoneType' is not iterable). Replace it with the real OpenAI Whisper:

    pip uninstall -y whisper
    pip install openai-whisper

Then restart the Jupyter kernel and re-run.

In [ ]:
# ============================================================================
# 1) Imports, paths & global configuration
# ============================================================================
import gc
import re
import time
from pathlib import Path

# Env vars must be set before huggingface_hub / transformers are imported.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "deepagents" else Path.cwd()
CONFIG_DIR   = PROJECT_ROOT / "config"
MODELS_DIR   = PROJECT_ROOT / "models"
OUTPUT_DIR   = PROJECT_ROOT / "reports"
AUDIO_FILE_PATH = PROJECT_ROOT / "meetings_audio" / "09-15-2026-Council-Meeting.mp3"

MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Keep every model artefact inside <repo>/models so the project stays portable.
os.environ["HF_HOME"] = str(MODELS_DIR / "hf-hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from dotenv import load_dotenv
load_dotenv(CONFIG_DIR / ".env")   # provides HF_TOKEN for the gated pyannote models
HF_TOKEN = os.getenv("HF_TOKEN", None)

import torch
import whisper
from pyannote.audio import Pipeline
from docx import Document
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from transformers import AutoModelForCausalLM, AutoTokenizer

# ---------------------------------------------------------------------------
# Global configuration
# ---------------------------------------------------------------------------
WHISPER_MODEL_SIZE   = "large-v3"
WHISPER_CACHE_DIR    = MODELS_DIR / "whisper"
PYANNOTE_PIPELINE_ID = "pyannote/speaker-diarization-3.1"
LLM_MODEL_ID         = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DOC_TYPE      = "pdf"        # Options: 'docx' or 'pdf'

# False -> download/open the open-source models from HuggingFace on first run
# (afterwards the HF cache under models/ is used without re-downloading).
# True  -> run 100% offline once everything has been cached locally.
RUN_OFFLINE = False

MAP_SUMMARY_TOKENS    = 512    # per-chunk output budget
REDUCE_SUMMARY_TOKENS = 1500   # final document budget

def free_vram():
    """Forces garbage collection and clears the PyTorch cache (CUDA/ROCm)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("[VRAM] Cache cleared.")

print(f"Project root : {PROJECT_ROOT}")
print(f"Models cache : {MODELS_DIR}")
print(f"Audio input  : {AUDIO_FILE_PATH}")
print("Environment initialized.")

In [ ]:
# ============================================================================
# 2) GPU sanity check (ROCm 10.1.0 on the RX 7900 XT reports as CUDA:True)
# ============================================================================
print(f"torch            : {torch.__version__}  ({torch.version.git_version[:8]})")
print(f"CUDA/ROCm runtime: {torch.version.hip or torch.version.cuda}")
gpu_ok = torch.cuda.is_available()
print(f"GPU available    : {gpu_ok}")
if gpu_ok:
    p = torch.cuda.get_device_properties(0)
    print(f"Device           : {p.name} ({p.gcnArchName})  VRAM: {p.total_memory / 1e9:.1f} GB")
    with torch.no_grad():
        assert (torch.zeros(64, device="cuda") @ torch.zeros(64, device="cuda")).shape == (64,)
    print("[OK] Small matmul on GPU succeeded.")
else:
    print("WARNING: no GPU detected - the pipeline falls back to CPU (very slow).")
    print("         Re-check the ROCm 10.1.0 install steps in the header cell.")

In [ ]:
# ============================================================================
# 3) One-time model download (run BEFORE switching to RUN_OFFLINE = True)
# ============================================================================
from huggingface_hub import snapshot_download

def download_all_models():
    """Downloads all open-source models from HuggingFace into models/ caches."""
    print(f\"\"\"[3b] Fetching diarization pipeline '{PYANNOTE_PIPELINE_ID}' ...\"\"\"\")
    snapshot_download(PYANNOTE_PIPELINE_ID, token=HF_TOKEN)

    print(f\"\"\"[3b] Fetching LLM '{LLM_MODEL_ID}' ...\"\"\"\")
    snapshot_download(LLM_MODEL_ID)

    print(f\"\"\"[3b] Fetching Whisper '{WHISPER_MODEL_SIZE}' via load_model ...\"\"\"\")
    m = whisper.load_model(WHISPER_MODEL_SIZE, device="cpu",
                           download_root=str(WHISPER_CACHE_DIR))
    del m
    free_vram()
    print("[3b] All models cached.")

In [ ]:
# ============================================================================
# 4) Stage 1/5 - Speaker diarization (PyAnnote)
# ============================================================================
_PIPELINES = {}

def _pipeline_from_pretrained(model_id: str):
    """Handles both the new (token=) and legacy (use_auth_token=) pyannote APIs."""
    last_err = None
    for kw in ({"token": HF_TOKEN}, {"use_auth_token": HF_TOKEN}):
        if RUN_OFFLINE:
            kw = {**kw, "local_files_only": True}
        try:
            return Pipeline.from_pretrained(model_id, **kw)
        except TypeError as e:
            last_err = e
    raise last_err

def diarize_audio(audio_path: str):
    """Stage 1: detect speaker turn intervals using PyAnnote."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n[1/5] Loading PyAnnote diarization pipeline -> {device}")

    if "diarization" not in _PIPELINES:
        _PIPELINES["diarization"] = _pipeline_from_pretrained(PYANNOTE_PIPELINE_ID)
        _PIPELINES["diarization"].to(device)
    pipeline = _PIPELINES["diarization"]

    print(f"[1/5] Identifying speakers in '{audio_path}' ...")
    started = time.time()
    diarization_result = pipeline(audio_path)
    print(f"[1/5] Done in {time.time() - started:.1f}s "
          f"({len(set(diarization_result.labels()))} speaker(s) detected)")
    free_vram()
    return diarization_result

In [ ]:
# ============================================================================
# 5) Stage 2/5 - Transcription (Whisper) + speaker alignment
# ============================================================================
def _fmt_ts(seconds: float) -> str:
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def transcribe_and_align(audio_path: str, diarization_result) -> str:
    """Stage 2: transcribe with Whisper, then attach speakers by time overlap."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n[2/5] Loading Whisper ({WHISPER_MODEL_SIZE}) -> {device}")
    model = whisper.load_model(WHISPER_MODEL_SIZE, device=device,
                               download_root=str(WHISPER_CACHE_DIR))

    print("[2/5] Transcribing audio ...")
    started = time.time()
    result = model.transcribe(
        str(audio_path),
        verbose=False,
        fp16=(device == "cuda"),
        condition_on_previous_text=False,  # prevents repetitive drift on long meetings
    )
    whisper_segments = result["segments"]
    print(f"[2/5] {len(whisper_segments)} segments transcribed "
          f"in {time.time() - started / 1: .0f}s")
    del model
    free_vram()

    print("[2/5] Aligning speaker IDs with transcript segments ...")
    turns = []
    for segment in whisper_segments:
        seg_start, seg_end = segment["start"], segment["end"]
        seg_text = segment["text"].strip()
        if not seg_text:
            continue

        best_speaker, max_overlap = "SPEAKER_UNK", 0.0
        for turn, _, speaker in diarization_result.itertracks(yield_label=True):
            overlap = max(0.0, min(seg_end, turn.end) - max(seg_start, turn.start))
            if overlap > max_overlap:
                max_overlap, best_speaker = overlap, speaker

        # Merge consecutive chunks from the same speaker into readable turns.
        if turns and turns[-1]["speaker"] == best_speaker:
            turns[-1]["end"] = seg_end
            turns[-1]["text"] += " " + seg_text
        else:
            turns.append({"speaker": best_speaker, "start": seg_start,
                          "end": seg_end, "text": seg_text})

    return "\n".join(
        f"[{_fmt_ts(t['start'])} - {_fmt_ts(t['end'])}] {t['speaker']}: {t['text']}"
        for t in turns
    )

In [ ]:
# ============================================================================
# 6) Stage 3/5 - Interactive speaker name mapping
# ============================================================================
def map_speakers_interactively(transcript: str) -> str:
    """Interactive rename of generic speaker tags (SPEAKER_00 -> Sarah)."""
    unique_speakers = sorted(set(re.findall(r"SPEAKER_\d{2,}", transcript)))

    if not unique_speakers:
        print("[3/5] No standard speaker tags found to map - returning transcript as-is.")
        return transcript

    print("\n[3/5] Interactive Speaker Mapping")
    print("-" * 45)
    print("Transcript preview:")
    print(transcript[:600] + ("..." if len(transcript) > 600 else ""))
    print("-" * 45)
    print("Enter real names for detected speakers (press Enter to keep default):")

    for speaker in unique_speakers:
        real_name = input(f" -> Rename '{speaker}' to: ").strip()
        if real_name:
            transcript = re.sub(rf"\b{re.escape(speaker)}\b", real_name, transcript)

    print("[3/5] Speaker mapping complete.")
    return transcript

In [ ]:
# ============================================================================
# 7) Stage 4/5 - Local LLM Map-Reduce summarization
# ============================================================================
def chunk_text(text: str, max_chars: int = 6000, overlap: int = 500) -> list:
    """Split a long transcript into overlapping, sentence-aligned chunks.

    Fixes the original loop-hazard where `start = end - overlap` could
    stagnate/rewind forever on short chunks."""
    text = text.strip()
    chunks, start, n = [], 0, len(text)

    while start < n:
        end = min(start + max_chars, n)
        if end < n:
            window = text[start:end]
            for sep in ("\n\n", ". ", ".\n", "\n", " "):
                cut = window.rfind(sep)
                if cut >= int(max_chars * 0.5):          # never split in the first half
                    end = start + cut + len(sep)
                    break

        chunks.append(text[start:end].strip())
        if end >= n:
            break
        start = max(end - overlap, start + 1)            # guaranteed forward progress

    return chunks

def chat_generate(model, tokenizer, messages: list, max_new_tokens: int) -> str:
    """Single greedy generation pass with a chat template (no sampling)."""
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                             # deterministic, TC-optimized
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def _load_llm():
    """Loads Qwen in bf16/fp16. Compatible with both transformers 4.x and 5.x."""
    base_kwargs = dict(device_map="auto", local_files_only=RUN_OFFLINE)
    attempts = (
        {"dtype": torch.bfloat16, "attn_implementation": "sdpa"},
        {"dtype": torch.bfloat16},
        {"torch_dtype": torch.float16},                  # legacy transformers 4.x
    )
    last_err = None
    for extra in attempts:
        try:
            model = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, **base_kwargs, **extra)
            break
        except (TypeError, ValueError) as e:
            last_err = e
    else:
        raise last_err
    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, local_files_only=RUN_OFFLINE)
    model.eval()
    return model, tokenizer

MAP_PROMPT_TEMPLATE = (
    "You are an assistant analyzing a portion of a meeting transcript.\n"
    "Extract key points, decisions, financial/technical details, and action items "
    "from this segment. Attribute action items to the speakers mentioned.\n\n"
    "Segment:\n\n{text}"
)

REDUCE_PROMPT_TEMPLATE = (
    "You are an executive assistant. Synthesize these partial meeting notes into a "
    "cohesive master document in Markdown.\n\n"
    "Structure:\n"
    "1. Executive Summary\n"
    "2. Key Discussions & Decisions\n"
    "3. Financial, Technical & Strategic Highlights\n"
    "4. Action Items & Next Steps (attributed to specific speakers)\n\n"
    "Use '# ', '## ' and '- ' Markdown syntax so the document exporter renders it well.\n\n"
    "Partial Notes:\n\n{notes}"
)

def generate_summary(transcript: str) -> str:
    """Stage 4: Map-Reduce LLM summarization with the locally cached open-source model."""
    print(f"\n[4/5] Loading LLM ({LLM_MODEL_ID}) ...")
    started = time.time()
    model, tokenizer = _load_llm()
    print(f"[4/5] Model ready in {time.time() - started:.0f}s "
          f"(quantization-free {dict(model.hf_device_map).get('')} "
          f"weights on {list(set(model.hf_device_map.values()))})")

    # 1. MAP PHASE -------------------------------------------------------------
    chunks = chunk_text(transcript)
    chunk_summaries = []
    print(f"[4/5] Map phase: processing {len(chunks)} transcript chunk(s) ...")
    for i, chunk in enumerate(chunks, 1):
        print(f"      Summarizing chunk {i}/{len(chunks)} ...", flush=True)
        messages = [
            {"role": "system", "content": "You are a precise executive assistant."},
            {"role": "user", "content": MAP_PROMPT_TEMPLATE.format(text=chunk)},
        ]
        chunk_summaries.append(chat_generate(model, tokenizer, messages, MAP_SUMMARY_TOKENS))
        free_vram()

    # 2. REDUCE PHASE ----------------------------------------------------------
    print("[4/5] Reduce phase: synthesizing master summary ...")
    combined = "\n\n".join(f"--- Part {i+1} ---\n{s}" for i, s in enumerate(chunk_summaries))
    messages = [
        {"role": "system",
         "content": "You synthesize meeting transcripts into clear executive summaries."},
        {"role": "user", "content": REDUCE_PROMPT_TEMPLATE.format(notes=combined)},
    ]
    final_summary = chat_generate(model, tokenizer, messages, REDUCE_SUMMARY_TOKENS)

    del model, tokenizer
    _PIPELINES.clear()
    free_vram()
    return final_summary

In [ ]:
# ============================================================================
# 8) Stage 5/5 - Markdown exporting (DOCX / PDF)
# ============================================================================
_MD_CLEAN = re.compile(r"\*\*(.+?)\*\*")          # wrapper used to un-bold later

def _md_to_plain(line: str) -> str:
    """Strip inline markdown emphasis markers for clean plain-text rendering."""
    return re.sub(r"\*\*|__", "", line).replace("`", "")

def export_docx(text: str, output_path: str):
    """Stage 5: convert the Markdown summary into a Word document."""
    doc = Document()
    doc.add_heading("Meeting Summary Document", 0)

    for line in text.split("\n"):
        line = line.rstrip()
        if not line.strip():
            continue
        if line.startswith("#### "):
            doc.add_heading(_md_to_plain(line[5:]), level=4)
        elif line.startswith("### "):
            doc.add_heading(_md_to_plain(line[4:]), level=3)
        elif line.startswith("## "):
            doc.add_heading(_md_to_plain(line[3:]), level=2)
        elif line.startswith("# "):
            doc.add_heading(_md_to_plain(line[2:]), level=1)
        elif line.lstrip().startswith(("- ", "* ")):
            doc.add_paragraph(_md_to_plain(line.lstrip()[2:]), style="List Bullet")
        else:
            doc.add_paragraph(_md_to_plain(line.strip()))

    doc.save(output_path)
    print(f"[5/5] Saved Word document : {output_path}")

_HEADING_STYLES = {1: ("Heading1", 10, 6), 2: ("Heading2", 8, 4), 3: ("Heading3", 8, 4), 4: ("Heading3", 6, 3)}

def export_pdf(text: str, output_path: str):
    """Stage 5: convert the Markdown summary into a PDF document."""
    doc = SimpleDocTemplate(output_path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = [Paragraph("Meeting Summary Document", styles["Title"]), Spacer(1, 12)]

    for line in text.split("\n"):
        line = line.rstrip()
        if not line.strip():
            story.append(Spacer(1, 4))
            continue
        if line.startswith("#"):
            level = len(line) - len(line.lstrip("#"))
            level = min(level, 4)
            style, above, below = _HEADING_STYLES.get(level, _HEADING_STYLES[2])
            story.extend([Spacer(1, above), Paragraph(_md_to_plain(line.lstrip('# ').strip()), styles[style]), Spacer(1, below)])
        elif line.lstrip().startswith(("- ", "* ")):
            story.append(Paragraph("\u2022 " + _md_to_plain(line.lstrip()[2:]), styles["BodyText"]))
            story.append(Spacer(1, 3))
        else:
            story.append(Paragraph(_md_to_plain(line.strip()), styles["BodyText"]))
            story.append(Spacer(1, 4))

    doc.build(story)
    print(f"[5/5] Saved PDF document   : {output_path}")

In [ ]:
# ============================================================================
# 9) Pipeline execution
# ============================================================================
def run_pipeline(audio_path: Path):
    if not audio_path.exists():
        raise FileNotFoundError(f"Target audio file does not exist: {audio_path}")

    print("=== Pipeline Initiated ===")
    pipeline_started = time.time()

    diarization_data = diarize_audio(str(audio_path))                       # Stage 1
    raw_transcript = transcribe_and_align(str(audio_path), diarization_data)  # Stage 2
    named_transcript = map_speakers_interactively(raw_transcript)           # Stage 3
    summary_markdown = generate_summary(named_transcript)                   # Stage 4

    # Stage 5 - Export documents & keep raw artefacts for reuse
    stem = audio_path.stem
    (OUTPUT_DIR / f"{stem}_transcript.txt").write_text(named_transcript, encoding="utf-8")
    (OUTPUT_DIR / f"{stem}_summary.md").write_text(summary_markdown, encoding="utf-8")
    print(f"[5/5] Saved transcript    : {OUTPUT_DIR / f'{stem}_transcript.txt'}")
    print(f"[5/5] Saved summary (md)  : {OUTPUT_DIR / f'{stem}_summary.md'}")

    output_doc = OUTPUT_DIR / f"{stem}_Summary.{OUTPUT_DOC_TYPE}"
    if OUTPUT_DOC_TYPE == "docx":
        export_docx(summary_markdown, str(output_doc))
    else:
        export_pdf(summary_markdown, str(output_doc))

    print(f"\n=== Pipeline Complete in {time.time() - pipeline_started:.0f}s ===")
    return summary_markdown

summary_markdown = run_pipeline(AUDIO_FILE_PATH)